# Clinical Hemodynamics: Interactive Exploration

**Target Audience:** Medical students, residents, and clinicians learning cardiovascular physiology

**Learning Objectives:**
1. Understand the relationship between preload, afterload, and cardiac output
2. Visualize pressure-volume loops and their clinical significance
3. Explore the effects of common clinical interventions
4. Interpret Swan-Ganz catheter waveforms

**Clinical Context:**
You're on rounds in the CCU and your attending asks: "What happens to the pressure-volume loop when we give this patient with heart failure a dose of furosemide?" This notebook will help you answer that question and many more.

---

In [ ]:
# Import required libraries
import sys
sys.path.append('..')  # Add parent directory to path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from src.validation.metrics import compute_pv_loop_metrics
from src.validation.benchmarks import PhysiologicalBenchmarks

# Set up plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✓ Imports successful")
print("\nReady to explore clinical hemodynamics!")

## Part 1: The Frank-Starling Mechanism

### Clinical Scenario
**Patient:** 65-year-old with acute decompensated heart failure  
**Question:** How does changing preload (volume status) affect cardiac performance?

### Physiological Background
The Frank-Starling law states that the stroke volume of the heart increases in response to an increase in the volume of blood in the ventricles (preload), when all other factors remain constant.

**Clinical relevance:**
- **Low preload:** Dehydration, hemorrhage, diuresis
- **High preload:** Volume overload, heart failure
- **Optimal preload:** Maximizes cardiac output without pulmonary congestion

In [ ]:
def generate_pv_loop(edv, ees=2.0, ea=1.5, v0=30.0):
    """
    Generate a pressure-volume loop using time-varying elastance model.
    
    Args:
        edv: End-diastolic volume (mL)
        ees: End-systolic elastance (mmHg/mL) - contractility
        ea: Arterial elastance (mmHg/mL) - afterload
        v0: Dead volume (mL)
    
    Returns:
        volumes, pressures (arrays)
    """
    # Time-varying elastance (simplified)
    n_points = 100
    t = np.linspace(0, 1, n_points)
    
    # Elastance varies sinusoidally during cardiac cycle
    # Peak at systole (t=0.3), minimum at diastole
    e_min = 0.1  # Diastolic elastance
    e_max = ees  # Systolic elastance
    
    # Elastance function
    elastance = e_min + (e_max - e_min) * np.sin(np.pi * t)**2
    
    # Find ESV using ventricular-arterial coupling
    # At equilibrium: Ees(ESV - V0) = Ea(EDV - ESV)
    esv = (ees * v0 + ea * edv) / (ees + ea)
    
    # Volume trajectory
    volumes = np.zeros(n_points)
    for i, ti in enumerate(t):
        if ti < 0.3:  # Isovolumic contraction + ejection
            # Starts at EDV, decreases to ESV
            volumes[i] = edv - (edv - esv) * (ti / 0.3)
        elif ti < 0.5:  # Isovolumic relaxation
            volumes[i] = esv
        else:  # Filling
            # Returns to EDV
            volumes[i] = esv + (edv - esv) * ((ti - 0.5) / 0.5)
    
    # Pressure = E(t) * (V(t) - V0)
    pressures = elastance * (volumes - v0)
    
    # Ensure non-negative pressures
    pressures = np.maximum(pressures, 0)
    
    return volumes, pressures


# Generate loops for different preload states
edv_low = 100  # mL - dehydrated patient
edv_normal = 140  # mL - euvolemic
edv_high = 180  # mL - volume overloaded

v_low, p_low = generate_pv_loop(edv_low)
v_normal, p_normal = generate_pv_loop(edv_normal)
v_high, p_high = generate_pv_loop(edv_high)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left panel: PV loops
ax1.plot(v_low, p_low, 'b-', linewidth=2, label='Low Preload (Dehydrated)')
ax1.plot(v_normal, p_normal, 'g-', linewidth=2, label='Normal Preload')
ax1.plot(v_high, p_high, 'r-', linewidth=2, label='High Preload (Volume Overload)')

ax1.set_xlabel('LV Volume (mL)', fontsize=13, fontweight='bold')
ax1.set_ylabel('LV Pressure (mmHg)', fontsize=13, fontweight='bold')
ax1.set_title('Frank-Starling: Effect of Preload on PV Loops', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 200)
ax1.set_ylim(0, 150)

# Add annotations
ax1.annotate('Stroke Volume ↑', xy=(120, 70), fontsize=11, 
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Right panel: Stroke work comparison
metrics_low = compute_pv_loop_metrics(p_low.tolist(), v_low.tolist())
metrics_normal = compute_pv_loop_metrics(p_normal.tolist(), v_normal.tolist())
metrics_high = compute_pv_loop_metrics(p_high.tolist(), v_high.tolist())

categories = ['Low Preload', 'Normal', 'High Preload']
stroke_volumes = [
    metrics_low['stroke_volume_ml'],
    metrics_normal['stroke_volume_ml'],
    metrics_high['stroke_volume_ml']
]
ejection_fractions = [
    metrics_low['ejection_fraction_pct'],
    metrics_normal['ejection_fraction_pct'],
    metrics_high['ejection_fraction_pct']
]

x = np.arange(len(categories))
width = 0.35

bars1 = ax2.bar(x - width/2, stroke_volumes, width, label='Stroke Volume (mL)', color='steelblue')
bars2 = ax2.bar(x + width/2, ejection_fractions, width, label='Ejection Fraction (%)', color='coral')

ax2.set_xlabel('Preload State', fontsize=13, fontweight='bold')
ax2.set_ylabel('Value', fontsize=13, fontweight='bold')
ax2.set_title('Stroke Volume and Ejection Fraction vs Preload', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n📊 Clinical Interpretation:")
print("="*60)
print(f"Low Preload:    SV = {stroke_volumes[0]:.1f} mL, EF = {ejection_fractions[0]:.1f}%")
print(f"Normal Preload: SV = {stroke_volumes[1]:.1f} mL, EF = {ejection_fractions[1]:.1f}%")
print(f"High Preload:   SV = {stroke_volumes[2]:.1f} mL, EF = {ejection_fractions[2]:.1f}%")
print("\n💡 Key Point: As preload increases, stroke volume increases!")
print("   This is the Frank-Starling mechanism in action.")

### 🏥 Clinical Application

**What you just saw:**
- Increasing preload (EDV) shifts the PV loop to the right
- Stroke volume (width of loop) increases
- This is why we give IV fluids to hypotensive patients (within limits!)

**On the wards:**
- Patient with dehydration → low preload → low cardiac output
- Give IV fluids → increase preload → increase cardiac output
- BUT: In heart failure, too much preload → pulmonary edema

---

## Part 2: Afterload and Contractility

### Clinical Scenario
**Patient:** 70-year-old with hypertensive emergency (BP 220/120)  
**Question:** How does high afterload affect the heart? What happens when we give vasodilators?

### Physiological Background
- **Afterload:** The resistance the ventricle must overcome to eject blood
- **Clinical surrogates:** Systemic vascular resistance (SVR), systolic blood pressure
- **Effect:** High afterload → decreased stroke volume → heart works harder

In [ ]:
# Generate loops with different afterload
edv = 140  # Keep preload constant
ees = 2.0  # Keep contractility constant

# Vary afterload (arterial elastance)
ea_low = 1.0    # Low SVR (e.g., septic shock)
ea_normal = 1.5  # Normal
ea_high = 3.0   # High SVR (e.g., hypertensive crisis)

v_al_low, p_al_low = generate_pv_loop(edv, ees=ees, ea=ea_low)
v_al_normal, p_al_normal = generate_pv_loop(edv, ees=ees, ea=ea_normal)
v_al_high, p_al_high = generate_pv_loop(edv, ees=ees, ea=ea_high)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# PV loops
ax1.plot(v_al_low, p_al_low, 'b-', linewidth=2, label='Low Afterload (Vasodilated)')
ax1.plot(v_al_normal, p_al_normal, 'g-', linewidth=2, label='Normal Afterload')
ax1.plot(v_al_high, p_al_high, 'r-', linewidth=2, label='High Afterload (Hypertensive)')

ax1.set_xlabel('LV Volume (mL)', fontsize=13, fontweight='bold')
ax1.set_ylabel('LV Pressure (mmHg)', fontsize=13, fontweight='bold')
ax1.set_title('Effect of Afterload on PV Loops', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 200)
ax1.set_ylim(0, 200)

# Add ESPVR line
v_espvr = np.linspace(30, 180, 50)
p_espvr = ees * (v_espvr - 30)
ax1.plot(v_espvr, p_espvr, 'k--', linewidth=1.5, label='ESPVR (Contractility)', alpha=0.7)

ax1.annotate('ESPVR', xy=(100, 140), fontsize=11, rotation=45,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Metrics comparison
m_al_low = compute_pv_loop_metrics(p_al_low.tolist(), v_al_low.tolist())
m_al_normal = compute_pv_loop_metrics(p_al_normal.tolist(), v_al_normal.tolist())
m_al_high = compute_pv_loop_metrics(p_al_high.tolist(), v_al_high.tolist())

categories = ['Low Afterload', 'Normal', 'High Afterload']
stroke_work = [
    m_al_low['stroke_work_j'],
    m_al_normal['stroke_work_j'],
    m_al_high['stroke_work_j']
]
stroke_volumes_al = [
    m_al_low['stroke_volume_ml'],
    m_al_normal['stroke_volume_ml'],
    m_al_high['stroke_volume_ml']
]

x = np.arange(len(categories))
width = 0.35

bars1 = ax2.bar(x - width/2, stroke_volumes_al, width, label='Stroke Volume (mL)', color='steelblue')
# Scale stroke work to similar range for visualization
stroke_work_scaled = [w * 10 for w in stroke_work]
bars2 = ax2.bar(x + width/2, stroke_work_scaled, width, label='Stroke Work (J × 10)', color='coral')

ax2.set_xlabel('Afterload State', fontsize=13, fontweight='bold')
ax2.set_ylabel('Value', fontsize=13, fontweight='bold')
ax2.set_title('Stroke Volume and Work vs Afterload', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, rotation=15)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n📊 Clinical Interpretation:")
print("="*60)
print(f"Low Afterload:    SV = {stroke_volumes_al[0]:.1f} mL, Work = {stroke_work[0]:.2f} J")
print(f"Normal Afterload: SV = {stroke_volumes_al[1]:.1f} mL, Work = {stroke_work[1]:.2f} J")
print(f"High Afterload:   SV = {stroke_volumes_al[2]:.1f} mL, Work = {stroke_work[2]:.2f} J")
print("\n💡 Key Points:")
print("   1. High afterload → DECREASED stroke volume")
print("   2. High afterload → INCREASED myocardial work")
print("   3. This is why we use ACE inhibitors/ARBs in heart failure!")

### 🏥 Clinical Application

**What you just saw:**
- High afterload shifts the end-systolic point up and to the right
- Stroke volume decreases (loop becomes narrower)
- Myocardial oxygen demand increases (more work to overcome resistance)

**On the wards:**
- Hypertensive emergency → give IV nitroprusside (vasodilator)
- Reduces afterload → increases stroke volume → reduces myocardial work
- Chronic heart failure → ACE inhibitors reduce afterload chronically

---

## Part 3: Contractility Changes

### Clinical Scenario
**Patient:** Cardiogenic shock post-MI, receiving dobutamine infusion  
**Question:** How do inotropes change cardiac function?

### Physiological Background
- **Contractility:** Intrinsic ability of myocardium to generate force
- **Measured by:** End-systolic elastance (Ees), dP/dt max
- **Positive inotropes:** Dobutamine, epinephrine, milrinone
- **Negative inotropes:** Beta-blockers, heart failure

In [ ]:
# Generate loops with different contractility
edv = 140
ea = 1.5

ees_low = 1.0      # Reduced contractility (heart failure)
ees_normal = 2.0   # Normal
ees_high = 3.5     # Enhanced contractility (inotrope)

v_c_low, p_c_low = generate_pv_loop(edv, ees=ees_low, ea=ea)
v_c_normal, p_c_normal = generate_pv_loop(edv, ees=ees_normal, ea=ea)
v_c_high, p_c_high = generate_pv_loop(edv, ees=ees_high, ea=ea)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Top left: PV loops
ax1 = axes[0, 0]
ax1.plot(v_c_low, p_c_low, 'r-', linewidth=2.5, label='Low Contractility (Heart Failure)')
ax1.plot(v_c_normal, p_c_normal, 'g-', linewidth=2.5, label='Normal Contractility')
ax1.plot(v_c_high, p_c_high, 'b-', linewidth=2.5, label='High Contractility (Inotrope)')

# Add ESPVR lines
v_line = np.linspace(30, 180, 50)
ax1.plot(v_line, ees_low * (v_line - 30), 'r--', linewidth=1.5, alpha=0.6, label='ESPVR (low)')
ax1.plot(v_line, ees_normal * (v_line - 30), 'g--', linewidth=1.5, alpha=0.6, label='ESPVR (normal)')
ax1.plot(v_line, ees_high * (v_line - 30), 'b--', linewidth=1.5, alpha=0.6, label='ESPVR (high)')

ax1.set_xlabel('LV Volume (mL)', fontsize=13, fontweight='bold')
ax1.set_ylabel('LV Pressure (mmHg)', fontsize=13, fontweight='bold')
ax1.set_title('Effect of Contractility on PV Loops', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 200)
ax1.set_ylim(0, 250)

ax1.annotate('Steeper ESPVR\n= Higher\nContractility', xy=(120, 180), fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Top right: Metrics comparison
ax2 = axes[0, 1]
m_c_low = compute_pv_loop_metrics(p_c_low.tolist(), v_c_low.tolist())
m_c_normal = compute_pv_loop_metrics(p_c_normal.tolist(), v_c_normal.tolist())
m_c_high = compute_pv_loop_metrics(p_c_high.tolist(), v_c_high.tolist())

categories = ['Heart Failure', 'Normal', 'Inotrope']
ef_values = [
    m_c_low['ejection_fraction_pct'],
    m_c_normal['ejection_fraction_pct'],
    m_c_high['ejection_fraction_pct']
]
sv_values = [
    m_c_low['stroke_volume_ml'],
    m_c_normal['stroke_volume_ml'],
    m_c_high['stroke_volume_ml']
]

x = np.arange(len(categories))
width = 0.35

bars1 = ax2.bar(x - width/2, sv_values, width, label='Stroke Volume (mL)', color='steelblue')
bars2 = ax2.bar(x + width/2, ef_values, width, label='Ejection Fraction (%)', color='coral')

ax2.set_xlabel('Contractility State', fontsize=13, fontweight='bold')
ax2.set_ylabel('Value', fontsize=13, fontweight='bold')
ax2.set_title('Stroke Volume and EF vs Contractility', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Bottom left: Simulated pressure waveform
ax3 = axes[1, 0]
time_cardiac = np.linspace(0, 3, 300)  # 3 cardiac cycles
hr = 75  # bpm
cycle_duration = 60.0 / hr

# Generate pressure waveforms
pressures_over_time = []
for t in time_cardiac:
    phase = (t % cycle_duration) / cycle_duration
    idx = int(phase * len(p_c_normal))
    if idx >= len(p_c_normal):
        idx = len(p_c_normal) - 1
    pressures_over_time.append(p_c_normal[idx])

ax3.plot(time_cardiac, pressures_over_time, 'g-', linewidth=2)
ax3.set_xlabel('Time (seconds)', fontsize=13, fontweight='bold')
ax3.set_ylabel('LV Pressure (mmHg)', fontsize=13, fontweight='bold')
ax3.set_title('Left Ventricular Pressure Waveform', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 150)

# Annotate phases
ax3.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='End systole')
ax3.axvline(x=0.5, color='blue', linestyle='--', alpha=0.5, label='End diastole')
ax3.legend(fontsize=10)

# Bottom right: Clinical interpretation table
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

table_data = [
    ['Parameter', 'Heart Failure', 'Normal', 'Dobutamine'],
    ['Ees (mmHg/mL)', f'{ees_low:.1f}', f'{ees_normal:.1f}', f'{ees_high:.1f}'],
    ['SV (mL)', f"{sv_values[0]:.1f}", f"{sv_values[1]:.1f}", f"{sv_values[2]:.1f}"],
    ['EF (%)', f"{ef_values[0]:.1f}", f"{ef_values[1]:.1f}", f"{ef_values[2]:.1f}"],
    ['ESP (mmHg)', f"{m_c_low['esp_mmhg']:.1f}", f"{m_c_normal['esp_mmhg']:.1f}", f"{m_c_high['esp_mmhg']:.1f}"],
]

table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.3, 0.23, 0.23, 0.23])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header row
for i in range(4):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color code columns
for i in range(1, 5):
    table[(i, 1)].set_facecolor('#ffcccb')  # Heart failure - red
    table[(i, 2)].set_facecolor('#c8e6c9')  # Normal - green
    table[(i, 3)].set_facecolor('#bbdefb')  # Inotrope - blue

ax4.set_title('Clinical Metrics Summary', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n📊 Clinical Interpretation:")
print("="*60)
print("EFFECT OF CONTRACTILITY CHANGES:")
print(f"\nHeart Failure:  EF = {ef_values[0]:.1f}% (REDUCED - abnormal)")
print(f"Normal:         EF = {ef_values[1]:.1f}% (Normal range)")
print(f"Dobutamine:     EF = {ef_values[2]:.1f}% (ENHANCED)")
print("\n💡 Key Points:")
print("   1. Contractility changes the SLOPE of ESPVR")
print("   2. Inotropes increase contractility → better EF and SV")
print("   3. Use in cardiogenic shock to augment cardiac output")

## Part 4: Clinical Scenarios

Now let's apply what we've learned to real clinical scenarios you'll encounter on the wards.

In [ ]:
def simulate_clinical_scenario(scenario_name, edv, ees, ea):
    """
    Simulate a clinical scenario and return PV loop + metrics.
    """
    v, p = generate_pv_loop(edv, ees=ees, ea=ea)
    metrics = compute_pv_loop_metrics(p.tolist(), v.tolist())
    
    return v, p, metrics

# Define scenarios
scenarios = {
    'Normal': {'edv': 140, 'ees': 2.0, 'ea': 1.5},
    'Hypovolemic Shock': {'edv': 80, 'ees': 2.0, 'ea': 2.5},  # Low preload, high SVR
    'Cardiogenic Shock': {'edv': 180, 'ees': 0.8, 'ea': 2.0},  # High preload, low contractility
    'Septic Shock': {'edv': 160, 'ees': 1.2, 'ea': 0.8},  # High preload, low SVR, reduced contractility
    'After IV Fluids': {'edv': 150, 'ees': 2.0, 'ea': 2.0},  # Increased preload from baseline shock
    'After Vasopressor': {'edv': 140, 'ees': 2.0, 'ea': 2.5},  # Increased SVR
    'After Inotrope': {'edv': 140, 'ees': 3.0, 'ea': 1.5},  # Increased contractility
}

# Simulate all scenarios
results = {}
for name, params in scenarios.items():
    results[name] = simulate_clinical_scenario(name, **params)

# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Shock states
ax1 = axes[0, 0]
shock_states = ['Normal', 'Hypovolemic Shock', 'Cardiogenic Shock', 'Septic Shock']
colors = ['green', 'blue', 'red', 'orange']
for state, color in zip(shock_states, colors):
    v, p, _ = results[state]
    ax1.plot(v, p, linewidth=2.5, label=state, color=color)

ax1.set_xlabel('LV Volume (mL)', fontsize=12, fontweight='bold')
ax1.set_ylabel('LV Pressure (mmHg)', fontsize=12, fontweight='bold')
ax1.set_title('Different Types of Shock', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 200)
ax1.set_ylim(0, 200)

# Interventions comparison
ax2 = axes[0, 1]
interventions = ['Hypovolemic Shock', 'After IV Fluids']
for state in interventions:
    v, p, _ = results[state]
    ax2.plot(v, p, linewidth=2.5, label=state)

ax2.set_xlabel('LV Volume (mL)', fontsize=12, fontweight='bold')
ax2.set_ylabel('LV Pressure (mmHg)', fontsize=12, fontweight='bold')
ax2.set_title('Effect of IV Fluid Resuscitation', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 200)
ax2.set_ylim(0, 200)

ax2.annotate('Rightward shift\n(↑ Preload)', xy=(120, 100), fontsize=11,
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

# Comparison of all interventions
ax3 = axes[1, 0]
baseline = 'Cardiogenic Shock'
interventions_all = [baseline, 'After IV Fluids', 'After Vasopressor', 'After Inotrope']
for state in interventions_all:
    v, p, _ = results[state]
    ax3.plot(v, p, linewidth=2.5, label=state)

ax3.set_xlabel('LV Volume (mL)', fontsize=12, fontweight='bold')
ax3.set_ylabel('LV Pressure (mmHg)', fontsize=12, fontweight='bold')
ax3.set_title('Treatment Options for Cardiogenic Shock', fontsize=13, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, 200)
ax3.set_ylim(0, 200)

# Metrics table
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

table_data = [['Scenario', 'SV (mL)', 'EF (%)', 'SW (J)']]
for name in ['Normal', 'Hypovolemic Shock', 'Cardiogenic Shock', 'Septic Shock']:
    _, _, metrics = results[name]
    table_data.append([
        name,
        f"{metrics['stroke_volume_ml']:.1f}",
        f"{metrics['ejection_fraction_pct']:.1f}",
        f"{metrics['stroke_work_j']:.2f}"
    ])

table = ax4.table(cellText=table_data, cellLoc='center', loc='center',
                 colWidths=[0.35, 0.2, 0.2, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style
for i in range(4):
    table[(0, i)].set_facecolor('#2196F3')
    table[(0, i)].set_text_props(weight='bold', color='white')

ax4.set_title('Hemodynamic Metrics Comparison', fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n🏥 CLINICAL SCENARIOS SUMMARY")
print("="*70)
for name in ['Normal', 'Hypovolemic Shock', 'Cardiogenic Shock', 'Septic Shock']:
    _, _, m = results[name]
    print(f"\n{name}:")
    print(f"  SV = {m['stroke_volume_ml']:.1f} mL | EF = {m['ejection_fraction_pct']:.1f}% | Work = {m['stroke_work_j']:.2f} J")

print("\n💡 Key Clinical Pearls:")
print("   • Hypovolemic shock: SMALL loop (low preload)")
print("   • Cardiogenic shock: WIDE loop (high preload, low contractility)")
print("   • Septic shock: INTERMEDIATE (distributive, mixed picture)")
print("   • Treatment: Match intervention to the underlying problem!")

## Summary and Key Takeaways

### 🎯 What You've Learned

1. **Frank-Starling Mechanism**
   - ↑ Preload → ↑ Stroke Volume (within limits)
   - Clinical: IV fluids increase preload
   - Caveat: Too much preload in heart failure → pulmonary edema

2. **Afterload Effects**
   - ↑ Afterload → ↓ Stroke Volume, ↑ Myocardial Work
   - Clinical: Vasodilators reduce afterload (nitroprusside, ACE inhibitors)
   - Application: Hypertensive emergency, heart failure

3. **Contractility**
   - Changes slope of ESPVR
   - Positive inotropes: dobutamine, epinephrine
   - Application: Cardiogenic shock

4. **Shock States**
   - **Hypovolemic:** Low preload → small PV loop
   - **Cardiogenic:** Low contractility → wide, low EF loop
   - **Septic:** Low SVR + reduced contractility → complex

### 📚 Further Reading
- Guyton & Hall: Textbook of Medical Physiology (Chapter 20)
- Lilly: Pathophysiology of Heart Disease (Chapter 9)
- See docs/REFERENCES.md in repository for complete citations

### 🔬 Next Steps
- Try modifying the parameters yourself
- Explore other clinical scenarios
- See notebook 02 for Heart Rate Variability analysis
- See notebook 03 for Swan-Ganz catheter waveforms

---

**Questions? Feedback?**  
Open an issue on GitHub or contact the maintainers.

---
© 2025 Multi-Heart-Model Project | MIT License